# Carrier Allocation Data Consolidation

Scans Excel files in configured lakehouse folders, identifies allocation worksheets
matching the `{Carrier} Alloc YYYY.MM.DD` pattern, extracts the data rows
(skipping metadata headers and summary footers), and consolidates everything
into a single delta table.

**Key behaviors:**
- `_02. TO BE TRANSFERRED`: processes only root-level `.xlsx` files (ignores year subfolders)
- `Mills TO BE TRANSFERRED`: processes all `.xlsx` files recursively
- Files with carrier code `UHC` in their name are skipped
- Header row is detected dynamically by scanning for `MEMBER NAME` in column A
- Summary/footer rows below the data are excluded automatically
- All extracted columns are stored as nullable strings in the delta table for
  schema-stable consolidation across heterogeneous carrier files; cast to
  date / decimal in downstream queries as needed.

In [ ]:
# ============================================================
# CONFIGURATION - adjust these values for your environment
# ============================================================

# Local mount path to the Lakehouse Files section.
LAKEHOUSE_FILES_PATH = "/lakehouse/default/Files"

# Folders to scan for Excel files.
# Set "root_only" to True to skip subfolders (e.g., year folders).
SCAN_FOLDERS = [
    {"path": "_02. TO BE TRANSFERRED", "root_only": True},
    {"path": "Mills TO BE TRANSFERRED", "root_only": False},
]

# Delta table name - written to the default lakehouse Tables section.
# Change this to your desired table name.
DELTA_TABLE_NAME = "consolidated_carrier_alloc"

In [ ]:
import os
import re
import logging
import warnings
import unicodedata
from datetime import datetime, date

from openpyxl import load_workbook
import pandas as pd
from pyspark.sql.functions import current_timestamp

# openpyxl prints a UserWarning for the "Data Validation extension" on
# every workbook open; it is harmless and just clutters the log.
warnings.filterwarnings(
    "ignore",
    message="Data Validation extension is not supported and will be removed",
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("carrier_alloc")

In [ ]:
# ============================================================
# CONSTANTS
# ============================================================

# Regex to match allocation worksheet names: "{Carrier} Alloc YYYY.MM.DD"
ALLOC_SHEET_RE = re.compile(r"^(.+?)\s+Alloc\s+(\d{4}\.\d{2}\.\d{2})$")

# Carrier codes to skip - any file whose name contains one of these is excluded.
SKIP_CARRIER_CODES = {"UHC"}

# Specific (filename, sheet_name) pairs to skip during processing. Use this
# for individual sheets with malformed or unwanted data when you don't want
# to exclude the entire file. Both fields compare case-insensitively;
# filename must be the basename only (no directory path).
SKIP_SHEETS = {
    # ("TO BE TRANSFERRED   EXAMPLE - Alloc 2026.01.12 SH.xlsx", "EXAMPLE Alloc 2026.01.12"),
}

# Header marker - the value expected in column A of the header row.
HEADER_MARKER = "MEMBER NAME"

# Maximum rows to scan from the top when looking for the header.
MAX_HEADER_SCAN = 50

# Maps normalized (whitespace-collapsed, uppercase) column name variants to
# their canonical form. Applied in _normalize_raw_header before the per-sheet
# dedup pass in extract_sheet_data. Keys must use single spaces -- whitespace
# normalization runs first, so multi-space variants are already collapsed
# before this lookup is reached.
COLUMN_ALIASES = {
    # Alternate names used by some carrier files for the same logical column.
    "THE TRUSTED PROGRAM AGENT":    "The Trusted Program (Referring Agent)",
    "ALLOCATION REFERRING AGENT":   "Allocation (Referring Agent) 1",
    "ALLOCATION REFERRING AGENT 1": "Allocation (Referring Agent) 1",
    "ALLOCATION REFERRING AGENT 2": "Allocation (Referring Agent) 2",
    "ALLOCATION REFERRING AGENT 3": "Allocation (Referring Agent) 3",
    "ALLOCATION REFERRING AGENT 4": "Allocation (Referring Agent) 4",
    "ALLOCATION AGENT":             "Alocation Agent (Reporting)",
    # Names unique to specific carrier files that map to a shared canonical column.
    "START DATE":                   "EFFECTIVE DATE",
    "NET COMMISSIONS":              "NET PAY",
}

# Valid carrier codes (for reference / optional validation).
VALID_CARRIER_CODES = {
    "BLUE IL", "BLUE MI", "BLUE MT", "BLUE TN", "BLUE TX", "BLUE",
    "BLUE RM", "BLUE VB", "BND SP", "BND VB", "CENT", "CENT VB",
    "CIGN HS", "CIGN MS", "CLEA", "CLOV RIM", "COMB", "CONT",
    "CORE RM", "DEVO", "DPBR", "EMBL", "EXCE RIM", "FLOR", "GUAR",
    "GEOB", "GLIC", "GLOB", "HFIR", "HFIR RIM", "HEAL", "HIGH RIM",
    "HORI", "HUMA CV", "HUMA NSG", "HUMA RM", "HUMA TAIA",
    "HUMA TAIA OR", "IMPE RIM", "MEDI", "MOLI", "MWG", "MOO",
    "MOO PDP", "O&A", "PHYS", "REGE RIM", "REGE", "RITTIM", "SCAN",
    "SILV RIM", "SILV SMS", "SPAR", "TRAN", "TRUE", "TRUS", "UA",
    "UHC", "UHC CV", "UHC RM", "VB", "VNS RIM", "VSP", "WHA",
    "WELL CE", "WELL NSG", "WELL SP", "WELL CV", "WELL RM",
}

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================


def _is_valid_xlsx(filename):
    """Return True for real .xlsx files, False for temp/hidden files."""
    return filename.lower().endswith(".xlsx") and not filename.startswith("~$")


def discover_excel_files(base_path, folder_cfg):
    """Return a sorted list of .xlsx file paths in a lakehouse folder."""
    folder = os.path.join(base_path, folder_cfg["path"])
    if not os.path.isdir(folder):
        log.warning("Folder not found: %s", folder)
        return []

    files = []

    if folder_cfg.get("root_only", False):
        for name in os.listdir(folder):
            full = os.path.join(folder, name)
            if os.path.isfile(full) and _is_valid_xlsx(name):
                files.append(full)
    else:
        for dirpath, _, filenames in os.walk(folder):
            for name in filenames:
                if _is_valid_xlsx(name):
                    files.append(os.path.join(dirpath, name))

    return sorted(files)


def should_skip_file(filepath):
    """Return True if the filename contains a carrier code we want to skip."""
    name_upper = os.path.basename(filepath).upper()
    return any(code in name_upper for code in SKIP_CARRIER_CODES)


def should_skip_sheet(filename, sheet_name):
    """Return True if (filename, sheet_name) is in the SKIP_SHEETS configuration.

    Both filename and sheet name are compared case-insensitively. The filename
    should be the basename only (no directory path), matching what
    os.path.basename returns from the file discovery loop.
    """
    target = (filename.lower(), sheet_name.lower())
    return any((f.lower(), s.lower()) == target for f, s in SKIP_SHEETS)


def find_alloc_sheets(sheetnames):
    """Return (sheet_name, carrier_code, alloc_date) for matching sheets.

    Takes a list of sheet names rather than a workbook object so it can
    be called without keeping the workbook open.
    """
    results = []
    for name in sheetnames:
        m = ALLOC_SHEET_RE.match(name)
        if m:
            results.append((name, m.group(1).strip(), m.group(2)))
    return results


def _normalize_raw_header(value):
    """Normalize a raw Excel header cell value before the per-sheet dedup pass.

    Applies two transformations in order:

      1. Whitespace normalization -- strips leading/trailing whitespace and
         collapses any internal run of whitespace to a single space.
         Prevents spurious extra columns from header typos such as
         "THE TRUSTED PROGRAM  50%" (two spaces) vs the single-space form;
         both collapse to the same string before dedup runs.

      2. Alias resolution -- performs a case-insensitive lookup of the
         normalized value against COLUMN_ALIASES and returns the canonical
         name when a match is found. Reconciles names that differ across
         carrier files but represent the same logical column
         (e.g., "START DATE" -> "EFFECTIVE DATE").

    Returns None unchanged so that callers can fall back to a "_col_N"
    placeholder for empty/missing header cells.
    """
    if value is None:
        return None
    normalized = " ".join(str(value).split())  # strip + collapse whitespace
    return COLUMN_ALIASES.get(normalized.upper(), normalized)


def extract_sheet_data(ws):
    """Detect the header row and extract all data rows from a worksheet.

    Strategy:
        1. Scan rows from the top for "MEMBER NAME" in column A -> header row.
        2. Build unique column names from the header (duplicates get a suffix,
           unnamed columns get a "_col_N" placeholder).
        3. Collect every subsequent row whose column A is non-empty. Empty
           column-A rows are either blank spacers or summary/footer rows,
           so they are safely skipped.

    Returns:
        (header_row_excel, col_names, records)
            header_row_excel: 1-based Excel row number of the header,
                              or None if not found.
            col_names:        list of unique column names.
            records:          list of dicts (one per data row).
    """
    all_rows = list(ws.iter_rows(values_only=True))

    # --- Locate header row ---
    header_idx = None
    for i, row in enumerate(all_rows[:MAX_HEADER_SCAN]):
        if row and row[0] is not None:
            if str(row[0]).strip().upper() == HEADER_MARKER:
                header_idx = i
                break

    if header_idx is None:
        return None, [], []

    header = list(all_rows[header_idx])
    n_cols = len(header)

    # --- Build unique column names (normalize, then case-insensitive dedup) ---
    # Each raw header value is first passed through _normalize_raw_header, which
    # collapses internal whitespace and resolves known aliases against
    # COLUMN_ALIASES. This ensures typographic variants like
    # "THE TRUSTED PROGRAM  50%" (two spaces) and semantic aliases like
    # "START DATE" are unified to their canonical names before dedup runs.
    #
    # This is the first of two dedup passes. It handles *per-sheet* uniqueness:
    # it guarantees distinct dict keys when building the record dicts below
    # (dict(zip(...)) would otherwise silently drop duplicates) and resolves
    # same-sheet case collisions like "CARRIER" vs "Carrier" before they reach
    # Spark -- Spark/Delta is case-insensitive and would raise
    # COLUMN_ALREADY_EXISTS on saveAsTable. Original casing is preserved on
    # the first occurrence; the second gets a "_2" suffix.
    # The second pass (post-sanitize, in sanitize_columns_for_delta) handles
    # *cross-sheet* collisions and any new collisions that arise from the
    # snake_case transformation itself -- see that function for details.
    col_names = []
    seen = {}  # key: lowercase name, value: occurrence count
    for j in range(n_cols):
        raw = header[j]
        normalized = _normalize_raw_header(raw)
        base = (
            normalized
            if normalized and str(normalized).strip()
            else f"_col_{j + 1}"
        )
        key = base.lower()
        if key not in seen:
            seen[key] = 1
            col_names.append(base)
        else:
            seen[key] += 1
            col_names.append(f"{base}_{seen[key]}")

    # --- Collect data rows ---
    records = []
    for row in all_rows[header_idx + 1:]:
        if not row or row[0] is None or str(row[0]).strip() == "":
            continue  # skip blank / summary rows

        padded = list(row)[:n_cols]
        padded += [None] * (n_cols - len(padded))
        records.append(dict(zip(col_names, padded)))

    return header_idx + 1, col_names, records  # 1-based Excel row


# ------------------------------------------------------------
# Column-name sanitization for Delta tables
# ------------------------------------------------------------
# Delta only forbids space and ',;{}()\n\t=' in column names by default,
# but we apply a stricter portable SQL identifier policy: lowercase ASCII
# snake_case. This makes column names portable across Delta / Parquet /
# Snowflake / BigQuery / Hive / Postgres and safe for dbt-style downstream
# queries, at the cost of rewriting headers like "HOUSE (RM) Upline 1" to
# "house_rm_upline_1". Collisions that arise from the transformation are
# resolved in a second dedup pass in sanitize_columns_for_delta.

# Matches any char that isn't lowercase ASCII, digit, or underscore.
# Despite the "identifier" framing this is stricter than any single
# warehouse requires -- it's the intersection of what's safe everywhere.
_NON_IDENT_CHARS_RE = re.compile(r"[^a-z0-9_]+")

# Collapses runs of underscores. This is NOT redundant with _NON_IDENT_CHARS_RE:
# that regex only collapses runs of *invalid* chars (so "foo   bar" -> "foo_bar"
# in one pass), but underscores are valid and pass through untouched. If the
# raw header already contains "foo__bar", the first regex leaves it alone and
# this second pass is what collapses it to "foo_bar".
_MULTI_UNDERSCORE_RE = re.compile(r"_+")

# Portable length cap: Snowflake 255, BigQuery 300, Hive 767 bytes.
_MAX_COL_NAME_LEN = 250


def _sanitize_one(name, position):
    """Convert a raw column name to a portable snake_case SQL identifier.

    Policy (aligned with Spark SQL unquoted-identifier rules and dbt
    naming conventions):

        1. NFKD-normalize unicode, drop combining marks (strip accents)
        2. Lowercase
        3. Replace any char not in [a-z0-9_] with underscore
        4. Collapse runs of underscores
        5. Strip leading/trailing underscores
        6. Fall back to 'col_{position}' if the result is empty
        7. Prefix with 'col_' if the result starts with a digit
        8. Truncate to _MAX_COL_NAME_LEN chars
    """
    if name is None:
        return f"col_{position}"

    raw = str(name)

    # 1. Unicode normalize + strip combining marks
    norm = unicodedata.normalize("NFKD", raw)
    norm = "".join(c for c in norm if not unicodedata.combining(c))

    # 2. Lowercase
    norm = norm.lower()

    # 3-4. Replace disallowed chars, then collapse any pre-existing
    # underscore runs that the first regex left alone.
    norm = _NON_IDENT_CHARS_RE.sub("_", norm)
    norm = _MULTI_UNDERSCORE_RE.sub("_", norm)

    # 5. Strip leading/trailing underscores
    norm = norm.strip("_")

    # 6. Empty fallback
    if not norm:
        return f"col_{position}"

    # 7. Digit-start fix (SQL unquoted identifiers must start with letter/_)
    if norm[0].isdigit():
        norm = f"col_{norm}"

    # 8. Portable length cap
    if len(norm) > _MAX_COL_NAME_LEN:
        norm = norm[:_MAX_COL_NAME_LEN].rstrip("_") or f"col_{position}"

    return norm


def sanitize_columns_for_delta(original_names, keep_as_is=None):
    """Sanitize and deduplicate a list of column names for a Delta table.

    Applies _sanitize_one to each name, preserves any names in keep_as_is
    verbatim (e.g., already-safe metadata columns that intentionally start
    with an underscore), and resolves any collisions that arise from the
    sanitization by suffixing duplicates with _2, _3, ...

    This is the *second* of two dedup passes. The first pass in
    extract_sheet_data handles per-sheet uniqueness and same-sheet case
    collisions on raw headers. This pass is what catches:
      - cross-sheet collisions (Sheet A's "CARRIER" vs Sheet B's "Carrier")
      - new collisions introduced by the snake_case transformation itself
        (e.g., "Member Name" and "MEMBER_NAME" both collapsing to "member_name")
    Both passes are needed; neither subsumes the other.

    Args:
        original_names: iterable of raw column names.
        keep_as_is:     optional set of names to pass through unchanged.

    Returns:
        A list of Delta-safe column names, same length as the input.
    """
    keep_as_is = set(keep_as_is or ())

    # First pass: sanitize or pass through
    renamed = []
    for i, name in enumerate(original_names):
        if name in keep_as_is:
            renamed.append(name)
        else:
            renamed.append(_sanitize_one(name, i + 1))

    # Second pass: resolve collisions with _N suffix
    seen = {}
    out = []
    for r in renamed:
        if r in seen:
            seen[r] += 1
            out.append(f"{r}_{seen[r]}")
        else:
            seen[r] = 1
            out.append(r)
    return out

In [ ]:
# ============================================================
# MAIN PROCESSING
# ============================================================

all_records = []
stats = {
    "files_found": 0,
    "files_skipped_uhc": 0,
    "sheets_matched": 0,
    "sheets_skipped": 0,
    "rows_extracted": 0,
    "errors": 0,
}

for cfg in SCAN_FOLDERS:
    files = discover_excel_files(LAKEHOUSE_FILES_PATH, cfg)
    log.info("Folder '%s' -- %d Excel file(s) found", cfg["path"], len(files))
    stats["files_found"] += len(files)

    for fpath in files:
        fname = os.path.basename(fpath)

        if should_skip_file(fpath):
            log.info("  SKIP  %s  (UHC carrier)", fname)
            stats["files_skipped_uhc"] += 1
            continue

        try:
            wb = load_workbook(fpath, read_only=True, data_only=True)
        except Exception as exc:
            log.error("  ERROR opening %s: %s", fname, exc)
            stats["errors"] += 1
            continue

        try:
            alloc_sheets = find_alloc_sheets(wb.sheetnames)
            if not alloc_sheets:
                log.debug("  No alloc sheets in %s", fname)
                continue

            for sheet_name, carrier_code, alloc_date in alloc_sheets:
                if should_skip_sheet(fname, sheet_name):
                    log.info(
                        "  SKIP  sheet '%s' in %s  (configured skip)",
                        sheet_name, fname,
                    )
                    stats["sheets_skipped"] += 1
                    continue

                ws = wb[sheet_name]
                _, _, records = extract_sheet_data(ws)

                if not records:
                    log.warning(
                        "  No data rows in '%s' (%s)", sheet_name, fname
                    )
                    continue

                # Attach metadata columns to each record.
                alloc_date_iso = alloc_date.replace(".", "-")
                for rec in records:
                    rec["_source_file"] = fname
                    rec["_sheet_name"] = sheet_name
                    rec["_carrier_code"] = carrier_code
                    rec["_alloc_date"] = alloc_date_iso

                all_records.extend(records)
                stats["sheets_matched"] += 1
                stats["rows_extracted"] += len(records)
                log.info(
                    "  >> '%s' -- %d rows extracted", sheet_name, len(records)
                )
        finally:
            wb.close()

log.info("Processing complete: %s", stats)

In [ ]:
# ============================================================
# SAVE TO DELTA TABLE
# ============================================================

if not all_records:
    log.warning("No records extracted. Delta table will not be created.")
else:
    pdf = pd.DataFrame(all_records)

    # Move metadata columns to the end for readability.
    meta_cols = ["_source_file", "_sheet_name", "_carrier_code", "_alloc_date"]
    data_cols = [c for c in pdf.columns if c not in meta_cols]
    pdf = pdf[data_cols + meta_cols]

    # ----- Normalize types for reliable Spark conversion -----
    # Excel cells in the same column position are inconsistently typed
    # across rows and across carrier files: a "PREMIUM PAID" cell may be
    # a float in one row, "" in another, and None in a third. The Arrow
    # converter that backs spark.createDataFrame raises ArrowInvalid on
    # such mixed object columns. Coercing every object-dtype column to a
    # nullable string is the safest universal cast - it preserves the value,
    # allows nulls, produces a stable schema across all carrier files, and
    # avoids NullType columns when a column happens to be entirely empty.
    # Cast columns to date / decimal in downstream queries as needed.
    def _to_str_or_null(v):
        if v is None:
            return None
        if isinstance(v, float) and pd.isna(v):
            return None
        return str(v)

    for col in pdf.columns:
        if pdf[col].dtype == object:
            pdf[col] = pdf[col].map(_to_str_or_null).astype("string")

    # ----- Convert to Spark -----
    sdf = spark.createDataFrame(pdf)

    # ----- Sanitize column names to snake_case for Delta -----
    # Delta only forbids space and ',;{}()\n\t=' by default, but we apply
    # a stricter portable SQL identifier policy (lowercase ASCII snake_case)
    # so column names are portable across Delta / Snowflake / BigQuery and
    # safe for dbt-style downstream queries. Metadata columns (leading _)
    # are intentional markers and pass through unchanged. See
    # sanitize_columns_for_delta in the helpers cell for the full policy.
    delta_safe_names = sanitize_columns_for_delta(
        sdf.columns,
        keep_as_is=set(meta_cols),
    )
    sdf = sdf.toDF(*delta_safe_names)

    # ----- Add ingestion timestamp and write -----
    sdf = sdf.withColumn("_ingested_at", current_timestamp())

    (
        sdf.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(DELTA_TABLE_NAME)
    )

    log.info(
        "Saved %d rows x %d columns to delta table '%s'",
        sdf.count(),
        len(sdf.columns),
        DELTA_TABLE_NAME,
    )

In [ ]:
# ============================================================
# VERIFY RESULTS
# ============================================================

df = spark.read.table(DELTA_TABLE_NAME)
print(f"Table:   {DELTA_TABLE_NAME}")
print(f"Rows:    {df.count()}")
print(f"Columns: {len(df.columns)}")
print()
display(df.limit(20))